# Apache Spark — Hands-on Lab & Answer Key
### RDD • PySpark DataFrame • Spark SQL

**Audience:** Beginner Spark learners  
**Format:** Each question is a Markdown cell followed by an executable solution code cell.  
**Duration:** ~2–3 hours (guided) or longer as an individual assignment.

The lab follows the topics in the supplied `spark_basics.py`: RDD transformations/actions, pair RDDs, joins, caching; DataFrame selection, filtering, computed columns, aggregations, windows, joins and null/dedup; Spark SQL WHERE, GROUP BY/HAVING, JOIN, WINDOW, CASE and CTE.

**Instructions:** Run the setup and data cells first. For each task, try your own code *before* looking at the next solution. Answers use only the sample data built in this notebook, so no external input files are required. Spark/PySpark must be installed.

## 0 — Environment setup

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("Spark_HandsOn_Assignment")
         .master("local[*]")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Default parallelism:", sc.defaultParallelism)

## 0.1 — Create the sample data
The **RDD sales dataset** is `(City, Category, Amount)`. The **DataFrame employee dataset** follows your original fields `(EmployeeID, Name, Department, Salary, City)`; extra records deliberately include duplicates and null values to practise cleaning.

In [ ]:
numbers_rdd = sc.parallelize(list(range(1, 11)), numSlices=4)

sales_data = [
    ("Cairo", "Electronics", 1200),
    ("Cairo", "Furniture", 800),
    ("Dubai", "Electronics", 2200),
    ("Dubai", "Furniture", 1700),
    ("Riyadh", "Electronics", 1500),
    ("Cairo", "Electronics", 500),
    ("Riyadh", "Furniture", 700),
    ("Dubai", "Electronics", 600),
]
sales_rdd = sc.parallelize(sales_data, numSlices=3)
region_rdd = sc.parallelize([
    ("Cairo", "EG"), ("Dubai", "UAE"), ("Riyadh", "KSA")
])

employee_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Salary", DoubleType(), True),
    StructField("City", StringType(), True),
])
employee_data = [
    (1, "Ahmed", "Engineering", 15000.0, "Cairo"),
    (2, "Sara", "Finance", 12000.0, "Dubai"),
    (3, "Mostafa", "Engineering", 18000.0, "Cairo"),
    (4, "Laila", "HR", 9000.0, "Riyadh"),
    (5, "Omar", "Finance", 13500.0, "Dubai"),
    (6, "Nour", "Engineering", 16500.0, "Riyadh"),
    (7, "Mona", "HR", None, "Cairo"),
    (8, "Youssef", "Finance", 14000.0, None),
]
employees_df = spark.createDataFrame(employee_data, employee_schema)
city_region_df = spark.createDataFrame([
    ("Cairo", "EG"), ("Dubai", "UAE"), ("Riyadh", "KSA")
], ["City", "Region"])

employees_df.createOrReplaceTempView("employees")
city_region_df.createOrReplaceTempView("city_region")

print("Numbers partitions:", numbers_rdd.getNumPartitions())
print("Sales rows:", sales_rdd.count())
employees_df.show(truncate=False)
city_region_df.show()

---
# Part 1 — RDD (30 points)
Each question is worth **5 points**. Practise transformations vs actions, `map`, `filter`, `flatMap`, pair RDD grouping, sorting, and joins.

### RDD.1 — map + filter

**Task:** Using `numbers_rdd`, multiply all numbers by **3**, then keep only those **greater than 15**. Display the result with `collect()`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### RDD.2 — flatMap word split

**Task:** Create an RDD from `["hello spark", "spark rdd basics", "hello data"]`. Use `flatMap()` to produce **one word per element**, and compare it with `map()`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### RDD.3 — RDD actions

**Task:** For `numbers_rdd`, calculate: total elements with `count()`, first three elements using `take(3)`, and sum using `reduce()`. Explain why `collect()` should be used cautiously for large data.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### RDD.4 — reduceByKey and sortBy

**Task:** Calculate **total sales amount per city** using a pair RDD and `reduceByKey()`, sorted from highest to lowest total.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### RDD.5 — groupByKey and join

**Task:** Display **all individual sales amounts** for each city using `groupByKey()`. Then join **city totals** to `region_rdd` to show `(City, (Total, Region))`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### RDD.6 — Partitions and cache

**Task:** Show the number of partitions in `numbers_rdd`, cache `city_totals`, and execute **two actions** (`count` and `collect`) on the cached RDD.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


---
# Part 2 — PySpark DataFrame (35 points)
Seven questions, **5 points each**. Use the `employees_df` and `city_region_df` datasets.

### DF.1 — Schema, select and filter

**Task:** Print the employee schema. Select `Name`, `Department`, and `Salary`, keeping employees with non-null salary **greater than 13,000**.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### DF.2 — withColumn bonus

**Task:** Create `Bonus` equal to **10% of Salary**, and `TotalPay` equal to salary plus bonus. Display the new columns; preserve null values.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### DF.3 — GroupBy aggregation

**Task:** For every department calculate `EmployeeCount`, `AvgSalary`, `MaxSalary`, and `MinSalary`. Sort by descending average salary. Remember that `avg` ignores null salaries.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### DF.4 — Window rank and running average

**Task:** Within each department, rank employees from highest to lowest salary using `rank()`, and calculate a running salary average ordered by descending salary. Treat null salary as last.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### DF.5 — Left join city-region

**Task:** Left join `employees_df` to `city_region_df` on `City`, retaining all employees even when their City is missing.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### DF.6 — Null handling

**Task:** Replace missing salaries with **0.0** and missing cities with `"Unknown"`. Count null salary records **before** and **after** filling.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### DF.7 — dropDuplicates vs distinct

**Task:** Create a small DataFrame with repeated rows. Show the difference between `distinct()` (whole row) and `dropDuplicates(["Department"])` (one record per department; other values are not deterministically selected).

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


---
# Part 3 — Spark SQL (35 points)
Seven questions, **5 points each**. Answer using `spark.sql()` over the temporary views `employees` and `city_region`.

### SQL.1 — SELECT / WHERE / ORDER BY

**Task:** Return the `Name`, `Department`, and `Salary` of employees earning **more than 13,000**, highest salary first.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### SQL.2 — GROUP BY + HAVING

**Task:** For each department with **at least 2 employees**, show the number of employees and their average **non-null** salary (rounded to 2 decimals). Sort by average salary descending.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### SQL.3 — LEFT JOIN

**Task:** Return `Name`, `City`, and `Region` for **all employees** using a LEFT JOIN between `employees` and `city_region`.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### SQL.4 — Window RANK and AVG

**Task:** Return `Name`, `Department`, `Salary`, a salary `RANK()` within each department, and each department’s average salary.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### SQL.5 — CASE WHEN salary band

**Task:** Create `SalaryBand` with these rules: salary **>= 16000 → High**, **>= 12000 → Medium**, otherwise **Low**; missing salary → **Unknown**.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### SQL.6 — CTE and above-department-average

**Task:** Using a `WITH` CTE, calculate department average salary, then return employees with salary **strictly greater** than their own department average. Include the amount above average.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


### SQL.7 — CTE + JOIN + HAVING challenge

**Task:** Write a SQL query to find the **total payroll per region**, excluding employees with missing salaries or unknown region. Return only regions with total payroll **greater than 20,000**.

<details><summary>Hint</summary>Try solving before running the answer below.</details>

In [ ]:
# Write your solution here


---
## Wrap-up and grading rubric

| Section | Questions | Marks |
|---|---:|---:|
| RDD | 6 | 30 |
| PySpark DataFrame | 7 | 35 |
| Spark SQL | 7 | 35 |
| **Total** | **20** | **100** |

**Suggested grading for each task (5 marks):** 3 for correct transformations/query logic, 1 for correct output, 1 for appropriate Spark API usage.  
**Discussion:** Why does `reduceByKey` generally scale better than `groupByKey` for sums? What triggers lazy execution? Why is `collect` risky? How do Spark SQL and DataFrame APIs relate?

**Optional exercise:** Implement one city-level sales summary in **all three APIs** and compare results.

In [ ]:
# Optional: compare the same sales totals in all three APIs
rdd_result = dict(sales_rdd.map(lambda r: (r[0], r[2])).reduceByKey(lambda a,b: a+b).collect())
sales_df = spark.createDataFrame(sales_data, ["City", "Category", "Amount"])
df_result = {r["City"]: r["Total"] for r in sales_df.groupBy("City").agg(F.sum("Amount").alias("Total")).collect()}
sales_df.createOrReplaceTempView("sales")
sql_result = {r["City"]: r["Total"] for r in spark.sql("SELECT City, SUM(Amount) AS Total FROM sales GROUP BY City").collect()}
print("RDD:", rdd_result)
print("DataFrame:", df_result)
print("SQL:", sql_result)
assert rdd_result == df_result == sql_result
print("✅ All three APIs give the same totals")

**End of lab.** Keep the Spark session alive if you want to explore the data further. Run `spark.stop()` when finished.